# Experiment 15: Detector Discrimination Under Anisotropic and Correlated Embedding Distributions

## 1. Objective

Investigate whether Euclidean and regularized Mahalanobis embedding-level drift detectors behave differently under anisotropic and correlated reference distributions, with controlled drift directions and independent calibration.

## 2. Research Questions

1. Does detector agreement change when reference dimensions have unequal variances?
2. How does drift direction relative to the covariance structure affect detection sensitivity?
3. Do Euclidean and Mahalanobis detectors achieve the nominal false-alarm target?
4. Are detector differences consistent across independent random seeds?

## 3. Experimental Conditions

- Isotropic reference distribution
- Anisotropic reference distribution
- Correlated reference distribution
- Drift along high-variance, low-variance, and mixed directions

## 4. Evaluation Metrics

- Detection rate
- False-alarm rate
- Detection delay
- Alarm-decision agreement
- Variability across independent seeds

## 5. Reproducibility

Document random seeds, covariance-generation parameters, calibration procedures, frozen thresholds, detector configurations, and output files. Preserve the reference, calibration, and evaluation data separation.

## 6. Scientific Limitations

Results from synthetic distributions do not establish real-world biometric recognition accuracy, fairness, security, or deployment readiness.

## 7. Relationship to Previous Experiments

Experiment 15 investigates detector behaviour under different covariance structures. It extends the temporal-pattern analysis of Experiment 14 without repeating its main research question.

In [1]:

# Experiment 15 — Step 2: Environment and Configuration

from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd

EXPERIMENT_ID = 15
BASE_SEED = 202615

OUTPUT_DIR = Path("/content/results/experiment_15")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment": EXPERIMENT_ID,
    "title": (
        "Detector Discrimination Under Anisotropic and "
        "Correlated Embedding Distributions"
    ),
    "base_seed": BASE_SEED,
    "n_seeds": 20,
    "embedding_dimension": 64,
    "window_size": 20,
    "n_windows": 60,
    "reference_samples": 1000,
    "calibration_windows": 200,
    "control_windows_per_seed": 500,
    "evaluation_windows_per_condition_per_seed": 500,
    "target_false_alarm_rate": 0.05,
    "mean_shift_strength": 0.5,
    "variance_profile": "geometric",
    "correlation_strength": 0.6,
    "drift_directions": [
        "high_variance",
        "low_variance",
        "mixed"
    ],
    "reference_conditions": [
        "isotropic",
        "anisotropic",
        "correlated"
    ],
    "detectors": [
        "euclidean",
        "mahalanobis"
    ],
    "threshold_method": "Empirical upper-tail quantile",
    "regularization": "LedoitWolf"
}

environment = {
    "experiment": EXPERIMENT_ID,
    "title": CONFIG["title"],
    "random_seed": BASE_SEED,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__
}

config_path = OUTPUT_DIR / "config.json"
environment_path = OUTPUT_DIR / "environment.json"

with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

with open(environment_path, "w") as f:
    json.dump(environment, f, indent=2)

print("Experiment:", EXPERIMENT_ID)
print("Output directory:", OUTPUT_DIR)
print("\nEnvironment:")
print(json.dumps(environment, indent=2))

print("\nConfiguration:")
print(json.dumps(CONFIG, indent=2))

assert CONFIG["embedding_dimension"] > 1
assert CONFIG["n_seeds"] >= 2
assert CONFIG["target_false_alarm_rate"] == 0.05
assert len(CONFIG["reference_conditions"]) == 3
assert len(CONFIG["drift_directions"]) == 3
assert len(CONFIG["detectors"]) == 2

print("\nConfiguration validation: PASSED")
print("Saved:", config_path)
print("Saved:", environment_path)


Experiment: 15
Output directory: /content/results/experiment_15

Environment:
{
  "experiment": 15,
  "title": "Detector Discrimination Under Anisotropic and Correlated Embedding Distributions",
  "random_seed": 202615,
  "python_version": "3.13.16",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "numpy_version": "2.1.3",
  "pandas_version": "2.2.3"
}

Configuration:
{
  "experiment": 15,
  "title": "Detector Discrimination Under Anisotropic and Correlated Embedding Distributions",
  "base_seed": 202615,
  "n_seeds": 20,
  "embedding_dimension": 64,
  "window_size": 20,
  "n_windows": 60,
  "reference_samples": 1000,
  "calibration_windows": 200,
  "control_windows_per_seed": 500,
  "evaluation_windows_per_condition_per_seed": 500,
  "target_false_alarm_rate": 0.05,
  "mean_shift_strength": 0.5,
  "variance_profile": "geometric",
  "correlation_strength": 0.6,
  "drift_directions": [
    "high_variance",
    "low_variance",
    "mixed"
  ],
  "reference_conditions": [
    "iso

In [2]:
# Experiment 15 — Step 3: Dependency and Numerical Capability Check

import numpy as np
import pandas as pd
import scipy
import sklearn

from scipy.linalg import eigh
from sklearn.covariance import LedoitWolf

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

# Verify Ledoit-Wolf covariance estimation
rng = np.random.default_rng(202615)
sample = rng.normal(size=(200, 64))

estimator = LedoitWolf()
estimator.fit(sample)

covariance = estimator.covariance_
eigenvalues = eigh(covariance, eigvals_only=True)

print("\nSample shape:", sample.shape)
print("Covariance shape:", covariance.shape)
print("All covariance values finite:", np.isfinite(covariance).all())
print("Covariance symmetric:", np.allclose(covariance, covariance.T))
print("Minimum covariance eigenvalue:", float(eigenvalues.min()))
print("Covariance positive definite:", bool(eigenvalues.min() > 0))

assert covariance.shape == (64, 64)
assert np.isfinite(covariance).all()
assert np.allclose(covariance, covariance.T)
assert eigenvalues.min() > 0

print("\nDependency and numerical checks: PASSED")

NumPy: 2.1.3
Pandas: 2.2.3
SciPy: 1.16.3
scikit-learn: 1.6.1

Sample shape: (200, 64)
Covariance shape: (64, 64)
All covariance values finite: True
Covariance symmetric: True
Minimum covariance eigenvalue: 0.9672278515623836
Covariance positive definite: True

Dependency and numerical checks: PASSED


In [3]:
# Experiment 15 — Step 4: Covariance Structure Validation

import numpy as np
import pandas as pd
from pathlib import Path

DIM = CONFIG["embedding_dimension"]
RHO = CONFIG["correlation_strength"]

# 1. Isotropic covariance: equal variance in every dimension
cov_isotropic = np.eye(DIM)

# 2. Anisotropic covariance: unequal variances across dimensions
variances = np.geomspace(0.25, 4.0, DIM)
cov_anisotropic = np.diag(variances)

# 3. Correlated covariance: equal marginal variances,
#    with correlation decreasing as dimensions get farther apart.
indices = np.arange(DIM)
cov_correlated = RHO ** np.abs(indices[:, None] - indices[None, :])

covariances = {
    "isotropic": cov_isotropic,
    "anisotropic": cov_anisotropic,
    "correlated": cov_correlated,
}

diagnostics = []

for condition, cov in covariances.items():
    eigenvalues = np.linalg.eigvalsh(cov)
    diagonal = np.diag(cov)

    diagnostics.append({
        "condition": condition,
        "dimension": cov.shape[0],
        "symmetric": bool(np.allclose(cov, cov.T)),
        "finite": bool(np.isfinite(cov).all()),
        "minimum_eigenvalue": float(eigenvalues.min()),
        "maximum_eigenvalue": float(eigenvalues.max()),
        "minimum_marginal_variance": float(diagonal.min()),
        "maximum_marginal_variance": float(diagonal.max()),
        "mean_absolute_off_diagonal": float(
            np.mean(np.abs(cov - np.diag(diagonal)))
        ),
    })

    assert cov.shape == (DIM, DIM)
    assert np.isfinite(cov).all()
    assert np.allclose(cov, cov.T)
    assert eigenvalues.min() > 0

diagnostics_df = pd.DataFrame(diagnostics)

diagnostics_path = OUTPUT_DIR / "covariance_diagnostics.csv"
diagnostics_df.to_csv(diagnostics_path, index=False)

print("Covariance diagnostics:")
print(diagnostics_df.to_string(index=False))

print("\nCovariance validation: PASSED")
print("Saved:", diagnostics_path)

Covariance diagnostics:
  condition  dimension  symmetric  finite  minimum_eigenvalue  maximum_eigenvalue  minimum_marginal_variance  maximum_marginal_variance  mean_absolute_off_diagonal
  isotropic         64       True    True             1.00000            1.000000                       1.00                        1.0                    0.000000
anisotropic         64       True    True             0.25000            4.000000                       0.25                        4.0                    0.000000
 correlated         64       True    True             0.25014            3.968234                       1.00                        1.0                    0.045044

Covariance validation: PASSED
Saved: /content/results/experiment_15/covariance_diagnostics.csv


In [4]:
# Experiment 15 — Step 5: Generate Reference Datasets

import numpy as np
import pandas as pd

DIM = CONFIG["embedding_dimension"]
N_REFERENCE = CONFIG["reference_samples"]
BASE_SEED = CONFIG["base_seed"]

reference_data = {}
reference_diagnostics = []

for condition_index, (condition, covariance) in enumerate(covariances.items()):
    seed = BASE_SEED + 100 + condition_index
    rng = np.random.default_rng(seed)

    samples = rng.multivariate_normal(
        mean=np.zeros(DIM),
        cov=covariance,
        size=N_REFERENCE,
        check_valid="raise"
    )

    empirical_covariance = np.cov(samples, rowvar=False)
    empirical_eigenvalues = np.linalg.eigvalsh(empirical_covariance)

    reference_data[condition] = samples

    reference_diagnostics.append({
        "condition": condition,
        "seed": seed,
        "samples": samples.shape[0],
        "dimensions": samples.shape[1],
        "all_finite": bool(np.isfinite(samples).all()),
        "empirical_mean_abs_average": float(
            np.abs(samples.mean(axis=0)).mean()
        ),
        "empirical_mean_variance": float(
            np.diag(empirical_covariance).mean()
        ),
        "empirical_min_eigenvalue": float(
            empirical_eigenvalues.min()
        ),
        "empirical_max_eigenvalue": float(
            empirical_eigenvalues.max()
        ),
    })

reference_diagnostics_df = pd.DataFrame(reference_diagnostics)

reference_path = OUTPUT_DIR / "reference_dataset_diagnostics.csv"
reference_diagnostics_df.to_csv(reference_path, index=False)

print("Reference dataset diagnostics:")
print(reference_diagnostics_df.to_string(index=False))

for condition, samples in reference_data.items():
    print(f"\n{condition}: shape={samples.shape}")

assert len(reference_data) == 3
assert all(
    data.shape == (N_REFERENCE, DIM)
    and np.isfinite(data).all()
    for data in reference_data.values()
)

print("\nReference dataset validation: PASSED")
print("Saved:", reference_path)

Reference dataset diagnostics:
  condition   seed  samples  dimensions  all_finite  empirical_mean_abs_average  empirical_mean_variance  empirical_min_eigenvalue  empirical_max_eigenvalue
  isotropic 202715     1000          64        True                    0.025545                 1.012654                  0.594452                  1.564769
anisotropic 202716     1000          64        True                    0.026534                 1.375044                  0.215101                  4.373390
 correlated 202717     1000          64        True                    0.028437                 1.006262                  0.191957                  4.252367

isotropic: shape=(1000, 64)

anisotropic: shape=(1000, 64)

correlated: shape=(1000, 64)

Reference dataset validation: PASSED
Saved: /content/results/experiment_15/reference_dataset_diagnostics.csv


In [5]:
# Experiment 15 — Step 6: Define and Validate Drift Directions

import numpy as np
import pandas as pd

DIM = CONFIG["embedding_dimension"]

drift_directions = {}
direction_diagnostics = []

for condition, covariance in covariances.items():
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)

    # Eigenvalues are sorted in ascending order.
    low_vector = eigenvectors[:, 0]
    high_vector = eigenvectors[:, -1]

    mixed_vector = high_vector + low_vector
    mixed_vector = mixed_vector / np.linalg.norm(mixed_vector)

    directions = {
        "low_variance": low_vector,
        "high_variance": high_vector,
        "mixed": mixed_vector,
    }

    drift_directions[condition] = directions

    for direction_name, vector in directions.items():
        vector_norm = float(np.linalg.norm(vector))
        projected_variance = float(vector @ covariance @ vector)

        direction_diagnostics.append({
            "condition": condition,
            "direction": direction_name,
            "vector_norm": vector_norm,
            "projected_population_variance": projected_variance,
            "all_finite": bool(np.isfinite(vector).all()),
        })

        assert np.isclose(vector_norm, 1.0)
        assert np.isfinite(vector).all()

direction_diagnostics_df = pd.DataFrame(direction_diagnostics)

direction_path = OUTPUT_DIR / "drift_direction_diagnostics.csv"
direction_diagnostics_df.to_csv(direction_path, index=False)

print("Drift direction diagnostics:")
print(direction_diagnostics_df.to_string(index=False))

print("\nDrift direction validation: PASSED")
print("Saved:", direction_path)

Drift direction diagnostics:
  condition     direction  vector_norm  projected_population_variance  all_finite
  isotropic  low_variance          1.0                       1.000000        True
  isotropic high_variance          1.0                       1.000000        True
  isotropic         mixed          1.0                       1.000000        True
anisotropic  low_variance          1.0                       0.250000        True
anisotropic high_variance          1.0                       4.000000        True
anisotropic         mixed          1.0                       2.125000        True
 correlated  low_variance          1.0                       0.250140        True
 correlated high_variance          1.0                       3.968234        True
 correlated         mixed          1.0                       2.109187        True

Drift direction validation: PASSED
Saved: /content/results/experiment_15/drift_direction_diagnostics.csv


In [6]:
# Experiment 15 — Step 7: Detector Scoring Functions

import numpy as np
from sklearn.covariance import LedoitWolf

def fit_detectors(reference_samples):
    """Fit reference statistics for both detectors."""
    reference_samples = np.asarray(reference_samples, dtype=float)

    if reference_samples.ndim != 2:
        raise ValueError("Reference samples must be a 2D array.")

    if not np.isfinite(reference_samples).all():
        raise ValueError("Reference samples contain non-finite values.")

    reference_mean = reference_samples.mean(axis=0)

    # Euclidean detector uses the reference mean.
    euclidean_model = {
        "mean": reference_mean.copy()
    }

    # Ledoit-Wolf provides a regularized covariance estimate.
    covariance_model = LedoitWolf().fit(reference_samples)
    covariance = covariance_model.covariance_

    # Cholesky factorization enables stable Mahalanobis calculations.
    cholesky_factor = np.linalg.cholesky(covariance)

    return {
        "euclidean": euclidean_model,
        "mahalanobis": {
            "mean": reference_mean.copy(),
            "cholesky": cholesky_factor,
            "covariance": covariance,
        },
    }


def score_windows(windows, fitted_models):
    """
    Score each window by the distance of its mean embedding
    from the reference mean.

    windows shape: (number_of_windows, window_size, dimensions)
    """
    windows = np.asarray(windows, dtype=float)

    if windows.ndim != 3:
        raise ValueError(
            "Windows must have shape (n_windows, window_size, dimensions)."
        )

    if not np.isfinite(windows).all():
        raise ValueError("Windows contain non-finite values.")

    window_means = windows.mean(axis=1)

    euclidean_delta = (
        window_means - fitted_models["euclidean"]["mean"]
    )
    euclidean_scores = np.linalg.norm(euclidean_delta, axis=1)

    mahalanobis_model = fitted_models["mahalanobis"]
    mahalanobis_delta = (
        window_means - mahalanobis_model["mean"]
    )

    # Solve Lz = delta.T, where covariance = L @ L.T.
    whitened_delta = np.linalg.solve(
        mahalanobis_model["cholesky"],
        mahalanobis_delta.T
    ).T

    mahalanobis_scores = np.sqrt(
        np.sum(whitened_delta ** 2, axis=1)
    )

    return {
        "euclidean": euclidean_scores,
        "mahalanobis": mahalanobis_scores,
    }


# Smoke test on one independent reference condition.
test_reference = reference_data["anisotropic"]
test_models = fit_detectors(test_reference)

test_rng = np.random.default_rng(CONFIG["base_seed"] + 999)
test_windows = test_rng.multivariate_normal(
    mean=np.zeros(DIM),
    cov=covariances["anisotropic"],
    size=CONFIG["window_size"] * 5
).reshape(5, CONFIG["window_size"], DIM)

test_scores = score_windows(test_windows, test_models)

for detector, scores in test_scores.items():
    print(
        f"{detector}: "
        f"shape={scores.shape}, "
        f"finite={np.isfinite(scores).all()}, "
        f"min={scores.min():.6f}, "
        f"max={scores.max():.6f}"
    )

assert all(scores.shape == (5,) for scores in test_scores.values())
assert all(np.isfinite(scores).all() for scores in test_scores.values())
assert all((scores >= 0).all() for scores in test_scores.values())

print("\nDetector scoring validation: PASSED")

euclidean: shape=(5,), finite=True, min=1.966079, max=2.136040
mahalanobis: shape=(5,), finite=True, min=1.614393, max=1.949562

Detector scoring validation: PASSED


In [7]:
# Experiment 15 — Step 8: Generate Independent Calibration Windows

import numpy as np
import pandas as pd

CALIBRATION_WINDOWS = CONFIG["calibration_windows"]
WINDOW_SIZE = CONFIG["window_size"]
DIM = CONFIG["embedding_dimension"]
BASE_SEED = CONFIG["base_seed"]

calibration_data = {}
calibration_models = {}
calibration_scores = {}
calibration_rows = []

for condition_index, (condition, covariance) in enumerate(covariances.items()):
    # Independent reference and calibration random streams.
    calibration_seed = BASE_SEED + 200 + condition_index
    rng = np.random.default_rng(calibration_seed)

    calibration_windows = rng.multivariate_normal(
        mean=np.zeros(DIM),
        cov=covariance,
        size=CALIBRATION_WINDOWS * WINDOW_SIZE,
        check_valid="raise"
    ).reshape(CALIBRATION_WINDOWS, WINDOW_SIZE, DIM)

    models = fit_detectors(reference_data[condition])
    scores = score_windows(calibration_windows, models)

    calibration_data[condition] = calibration_windows
    calibration_models[condition] = models
    calibration_scores[condition] = scores

    for detector, detector_scores in scores.items():
        for window_index, score in enumerate(detector_scores):
            calibration_rows.append({
                "condition": condition,
                "detector": detector,
                "calibration_seed": calibration_seed,
                "window_index": window_index,
                "score": float(score),
            })

calibration_scores_df = pd.DataFrame(calibration_rows)

calibration_path = OUTPUT_DIR / "calibration_scores.csv"
calibration_scores_df.to_csv(calibration_path, index=False)

print("Calibration windows by condition:")
for condition, windows in calibration_data.items():
    print(f"{condition}: {windows.shape}")

print("\nCalibration score summary:")
print(
    calibration_scores_df
    .groupby(["condition", "detector"])["score"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(6)
    .to_string()
)

assert len(calibration_data) == 3
assert all(
    windows.shape == (CALIBRATION_WINDOWS, WINDOW_SIZE, DIM)
    and np.isfinite(windows).all()
    for windows in calibration_data.values()
)
assert len(calibration_scores_df) == 3 * 2 * CALIBRATION_WINDOWS
assert np.isfinite(calibration_scores_df["score"]).all()

print("\nCalibration data validation: PASSED")
print("Saved:", calibration_path)

Calibration windows by condition:
isotropic: (200, 20, 64)
anisotropic: (200, 20, 64)
correlated: (200, 20, 64)

Calibration score summary:
                         count      mean       std       min       max
condition   detector                                                  
anisotropic euclidean      200  2.098745  0.231502  1.548152  2.940960
            mahalanobis    200  1.779366  0.153675  1.373607  2.238377
correlated  euclidean      200  1.820760  0.219760  1.302320  2.417135
            mahalanobis    200  1.797744  0.161988  1.390924  2.316308
isotropic   euclidean      200  1.812985  0.157642  1.430458  2.205591
            mahalanobis    200  1.802524  0.156733  1.422204  2.192864

Calibration data validation: PASSED
Saved: /content/results/experiment_15/calibration_scores.csv


In [8]:
# Experiment 15 — Step 9: Freeze Detector Thresholds

import json
import numpy as np
import pandas as pd

TARGET_FPR = CONFIG["target_false_alarm_rate"]

threshold_records = []
frozen_thresholds = {}

for condition in CONFIG["reference_conditions"]:
    frozen_thresholds[condition] = {}

    for detector in CONFIG["detectors"]:
        scores = calibration_scores[condition][detector]

        # "higher" selects an observed calibration score at or above
        # the requested empirical quantile where supported.
        try:
            threshold = np.quantile(
                scores, 1 - TARGET_FPR, method="higher"
            )
        except TypeError:
            threshold = np.quantile(
                scores, 1 - TARGET_FPR, interpolation="higher"
            )

        threshold = float(threshold)
        frozen_thresholds[condition][detector] = threshold

        threshold_records.append({
            "condition": condition,
            "detector": detector,
            "target_fpr": TARGET_FPR,
            "calibration_windows": len(scores),
            "threshold": threshold,
            "calibration_alarm_count": int(np.sum(scores > threshold)),
            "calibration_alarm_rate": float(np.mean(scores > threshold)),
        })

thresholds_df = pd.DataFrame(threshold_records)

threshold_path = OUTPUT_DIR / "frozen_thresholds.json"
threshold_table_path = OUTPUT_DIR / "frozen_thresholds.csv"

with open(threshold_path, "w") as f:
    json.dump(frozen_thresholds, f, indent=2)

thresholds_df.to_csv(threshold_table_path, index=False)

print("Frozen thresholds:")
print(thresholds_df.to_string(index=False))

print("\nThresholds saved to:", threshold_path)
print("Threshold table saved to:", threshold_table_path)

assert len(threshold_records) == 6
assert all(
    np.isfinite(record["threshold"])
    for record in threshold_records
)

print("\nThreshold validation: PASSED")

Frozen thresholds:
  condition    detector  target_fpr  calibration_windows  threshold  calibration_alarm_count  calibration_alarm_rate
  isotropic   euclidean        0.05                  200   2.089476                        9                   0.045
  isotropic mahalanobis        0.05                  200   2.077419                        9                   0.045
anisotropic   euclidean        0.05                  200   2.503961                        9                   0.045
anisotropic mahalanobis        0.05                  200   2.055089                        9                   0.045
 correlated   euclidean        0.05                  200   2.194061                        9                   0.045
 correlated mahalanobis        0.05                  200   2.059885                        9                   0.045

Thresholds saved to: /content/results/experiment_15/frozen_thresholds.json
Threshold table saved to: /content/results/experiment_15/frozen_thresholds.csv

Thresh

In [9]:
# Experiment 15 — Step 10: Independent Control Evaluation

import numpy as np
import pandas as pd

N_CONTROL = CONFIG["control_windows_per_seed"]
WINDOW_SIZE = CONFIG["window_size"]
DIM = CONFIG["embedding_dimension"]
BASE_SEED = CONFIG["base_seed"]

control_data = {}
control_rows = []

for condition_index, (condition, covariance) in enumerate(covariances.items()):
    control_seed = BASE_SEED + 300 + condition_index
    rng = np.random.default_rng(control_seed)

    windows = rng.multivariate_normal(
        mean=np.zeros(DIM),
        cov=covariance,
        size=N_CONTROL * WINDOW_SIZE,
        check_valid="raise"
    ).reshape(N_CONTROL, WINDOW_SIZE, DIM)

    scores = score_windows(windows, calibration_models[condition])
    control_data[condition] = windows

    for detector, detector_scores in scores.items():
        threshold = frozen_thresholds[condition][detector]
        alarms = detector_scores > threshold

        for window_index, (score, alarm) in enumerate(
            zip(detector_scores, alarms)
        ):
            control_rows.append({
                "condition": condition,
                "detector": detector,
                "control_seed": control_seed,
                "window_index": window_index,
                "score": float(score),
                "threshold": threshold,
                "alarm": bool(alarm),
            })

control_results_df = pd.DataFrame(control_rows)

control_path = OUTPUT_DIR / "independent_control_results.csv"
control_results_df.to_csv(control_path, index=False)

control_summary_df = (
    control_results_df
    .groupby(["condition", "detector"])
    .agg(
        control_windows=("alarm", "size"),
        false_alarms=("alarm", "sum"),
        observed_fpr=("alarm", "mean"),
        mean_score=("score", "mean"),
        score_sd=("score", "std"),
    )
    .reset_index()
)

summary_path = OUTPUT_DIR / "independent_control_summary.csv"
control_summary_df.to_csv(summary_path, index=False)

print("Independent control summary:")
print(control_summary_df.round(6).to_string(index=False))

assert len(control_results_df) == 3 * 2 * N_CONTROL
assert control_results_df["score"].map(np.isfinite).all()
assert control_results_df["threshold"].map(np.isfinite).all()
assert all(len(windows) == N_CONTROL for windows in control_data.values())

print("\nIndependent control validation: PASSED")
print("Saved:", control_path)
print("Saved:", summary_path)

Independent control summary:
  condition    detector  control_windows  false_alarms  observed_fpr  mean_score  score_sd
anisotropic   euclidean              500            17         0.034    2.079793  0.220906
anisotropic mahalanobis              500            15         0.030    1.768615  0.152694
 correlated   euclidean              500            23         0.046    1.795539  0.234243
 correlated mahalanobis              500            34         0.068    1.800006  0.165674
  isotropic   euclidean              500            20         0.040    1.793836  0.161553
  isotropic mahalanobis              500            20         0.040    1.783485  0.160621

Independent control validation: PASSED
Saved: /content/results/experiment_15/independent_control_results.csv
Saved: /content/results/experiment_15/independent_control_summary.csv


In [10]:
# Experiment 15 — Step 11: Generate Controlled Drift Evaluation Data

import numpy as np
import pandas as pd

N_EVAL = CONFIG["evaluation_windows_per_condition_per_seed"]
WINDOW_SIZE = CONFIG["window_size"]
DIM = CONFIG["embedding_dimension"]
SHIFT_STRENGTH = CONFIG["mean_shift_strength"]
BASE_SEED = CONFIG["base_seed"]

evaluation_data = {}
evaluation_rows = []

for condition_index, condition in enumerate(CONFIG["reference_conditions"]):
    covariance = covariances[condition]
    evaluation_data[condition] = {}

    for direction_index, direction_name in enumerate(CONFIG["drift_directions"]):
        seed = BASE_SEED + 400 + condition_index * 10 + direction_index
        rng = np.random.default_rng(seed)

        direction = drift_directions[condition][direction_name]

        # A fixed mean shift applied to every sample in every evaluation window.
        mean_shift = SHIFT_STRENGTH * direction

        windows = rng.multivariate_normal(
            mean=mean_shift,
            cov=covariance,
            size=N_EVAL * WINDOW_SIZE,
            check_valid="raise"
        ).reshape(N_EVAL, WINDOW_SIZE, DIM)

        evaluation_data[condition][direction_name] = windows

        models = calibration_models[condition]
        scores = score_windows(windows, models)

        for detector, detector_scores in scores.items():
            threshold = frozen_thresholds[condition][detector]
            alarms = detector_scores > threshold

            for window_index, (score, alarm) in enumerate(
                zip(detector_scores, alarms)
            ):
                evaluation_rows.append({
                    "condition": condition,
                    "direction": direction_name,
                    "detector": detector,
                    "seed": seed,
                    "window_index": window_index,
                    "shift_strength": SHIFT_STRENGTH,
                    "score": float(score),
                    "threshold": threshold,
                    "alarm": bool(alarm),
                })

evaluation_results_df = pd.DataFrame(evaluation_rows)

evaluation_path = OUTPUT_DIR / "drift_evaluation_results.csv"
evaluation_results_df.to_csv(evaluation_path, index=False)

evaluation_summary_df = (
    evaluation_results_df
    .groupby(["condition", "direction", "detector"])
    .agg(
        evaluation_windows=("alarm", "size"),
        detected_windows=("alarm", "sum"),
        detection_rate=("alarm", "mean"),
        mean_score=("score", "mean"),
        score_sd=("score", "std"),
    )
    .reset_index()
)

evaluation_summary_path = OUTPUT_DIR / "drift_evaluation_summary.csv"
evaluation_summary_df.to_csv(evaluation_summary_path, index=False)

print("Drift evaluation summary:")
print(evaluation_summary_df.round(6).to_string(index=False))

assert len(evaluation_results_df) == (
    len(CONFIG["reference_conditions"])
    * len(CONFIG["drift_directions"])
    * len(CONFIG["detectors"])
    * N_EVAL
)

assert evaluation_results_df["score"].map(np.isfinite).all()
assert evaluation_results_df["threshold"].map(np.isfinite).all()

print("\nDrift evaluation validation: PASSED")
print("Saved:", evaluation_path)
print("Saved:", evaluation_summary_path)

Drift evaluation summary:
  condition     direction    detector  evaluation_windows  detected_windows  detection_rate  mean_score  score_sd
anisotropic high_variance   euclidean                 500                38           0.076    2.131120  0.234191
anisotropic high_variance mahalanobis                 500                31           0.062    1.779909  0.163786
anisotropic  low_variance   euclidean                 500                33           0.066    2.141799  0.230014
anisotropic  low_variance mahalanobis                 500               140           0.280    1.962511  0.171159
anisotropic         mixed   euclidean                 500                34           0.068    2.143577  0.231310
anisotropic         mixed mahalanobis                 500                78           0.156    1.877054  0.166984
 correlated high_variance   euclidean                 500                43           0.086    1.856408  0.254728
 correlated high_variance mahalanobis                 500     

In [11]:
# Experiment 15 — Step 12: Detector Alarm Agreement

import numpy as np
import pandas as pd

agreement_rows = []

for condition in CONFIG["reference_conditions"]:
    for direction in CONFIG["drift_directions"]:
        subset = evaluation_results_df[
            (evaluation_results_df["condition"] == condition)
            & (evaluation_results_df["direction"] == direction)
        ]

        euclidean = (
            subset[subset["detector"] == "euclidean"]
            .sort_values("window_index")
        )
        mahalanobis = (
            subset[subset["detector"] == "mahalanobis"]
            .sort_values("window_index")
        )

        assert np.array_equal(
            euclidean["window_index"].to_numpy(),
            mahalanobis["window_index"].to_numpy()
        )

        e_alarm = euclidean["alarm"].to_numpy(dtype=bool)
        m_alarm = mahalanobis["alarm"].to_numpy(dtype=bool)

        agreement = float(np.mean(e_alarm == m_alarm))
        both_alarm = int(np.sum(e_alarm & m_alarm))
        euclidean_only = int(np.sum(e_alarm & ~m_alarm))
        mahalanobis_only = int(np.sum(~e_alarm & m_alarm))
        neither_alarm = int(np.sum(~e_alarm & ~m_alarm))

        agreement_rows.append({
            "condition": condition,
            "direction": direction,
            "evaluation_windows": len(e_alarm),
            "alarm_agreement_rate": agreement,
            "both_alarm": both_alarm,
            "euclidean_only": euclidean_only,
            "mahalanobis_only": mahalanobis_only,
            "neither_alarm": neither_alarm,
        })

agreement_df = pd.DataFrame(agreement_rows)

agreement_path = OUTPUT_DIR / "detector_alarm_agreement.csv"
agreement_df.to_csv(agreement_path, index=False)

print("Detector alarm agreement:")
print(agreement_df.round(6).to_string(index=False))

assert len(agreement_df) == 9
assert np.allclose(
    agreement_df["both_alarm"]
    + agreement_df["euclidean_only"]
    + agreement_df["mahalanobis_only"]
    + agreement_df["neither_alarm"],
    agreement_df["evaluation_windows"]
)

print("\nAlarm agreement validation: PASSED")
print("Saved:", agreement_path)

Detector alarm agreement:
  condition     direction  evaluation_windows  alarm_agreement_rate  both_alarm  euclidean_only  mahalanobis_only  neither_alarm
  isotropic high_variance                 500                 1.000          55               0                 0            445
  isotropic  low_variance                 500                 1.000          59               0                 0            441
  isotropic         mixed                 500                 1.000          52               0                 0            448
anisotropic high_variance                 500                 0.946          21              17                10            452
anisotropic  low_variance                 500                 0.770          29               4               111            356
anisotropic         mixed                 500                 0.880          26               8                52            414
 correlated high_variance                 500                 0.922    

In [12]:
# Experiment 15 — Step 13: Detection-Rate Differences

import pandas as pd
import numpy as np

rate_table = evaluation_summary_df.pivot(
    index=["condition", "direction"],
    columns="detector",
    values="detection_rate"
).reset_index()

rate_table["mahalanobis_minus_euclidean"] = (
    rate_table["mahalanobis"] - rate_table["euclidean"]
)

rate_table["absolute_difference"] = (
    rate_table["mahalanobis_minus_euclidean"].abs()
)

rate_table = rate_table.sort_values(
    ["condition", "direction"]
).reset_index(drop=True)

difference_path = OUTPUT_DIR / "detection_rate_differences.csv"
rate_table.to_csv(difference_path, index=False)

print("Detection-rate comparison:")
print(rate_table.round(4).to_string(index=False))

print("\nLargest absolute detection-rate differences:")
print(
    rate_table.nlargest(3, "absolute_difference")[
        [
            "condition",
            "direction",
            "euclidean",
            "mahalanobis",
            "mahalanobis_minus_euclidean",
        ]
    ].round(4).to_string(index=False)
)

assert len(rate_table) == 9
assert np.isfinite(
    rate_table["mahalanobis_minus_euclidean"]
).all()

print("\nDetection-rate comparison validation: PASSED")
print("Saved:", difference_path)

Detection-rate comparison:
  condition     direction  euclidean  mahalanobis  mahalanobis_minus_euclidean  absolute_difference
anisotropic high_variance      0.076        0.062                       -0.014                0.014
anisotropic  low_variance      0.066        0.280                        0.214                0.214
anisotropic         mixed      0.068        0.156                        0.088                0.088
 correlated high_variance      0.086        0.056                       -0.030                0.030
 correlated  low_variance      0.082        0.396                        0.314                0.314
 correlated         mixed      0.104        0.214                        0.110                0.110
  isotropic high_variance      0.110        0.110                        0.000                0.000
  isotropic  low_variance      0.118        0.118                        0.000                0.000
  isotropic         mixed      0.104        0.104                        

In [13]:
# Experiment 15 — Step 14: Prepare Multi-Seed Evaluation

import numpy as np
import pandas as pd

N_SEEDS = CONFIG["n_seeds"]

print("Configured seeds:", N_SEEDS)
print("Existing evaluation rows:", len(evaluation_results_df))
print("Existing evaluation seeds:", evaluation_results_df["seed"].nunique())

print("\nExisting evaluation rows per condition and direction:")
print(
    evaluation_results_df.groupby(
        ["condition", "direction", "detector"]
    ).size().to_string()
)

assert N_SEEDS == 20
assert evaluation_results_df["seed"].nunique() == 9
assert len(evaluation_results_df) == 9 * 2 * CONFIG[
    "evaluation_windows_per_condition_per_seed"
]

print("\nMulti-seed preparation check: PASSED")
print("Next: generate independent seed-level reference and calibration data.")

Configured seeds: 20
Existing evaluation rows: 9000
Existing evaluation seeds: 9

Existing evaluation rows per condition and direction:
condition    direction      detector   
anisotropic  high_variance  euclidean      500
                            mahalanobis    500
             low_variance   euclidean      500
                            mahalanobis    500
             mixed          euclidean      500
                            mahalanobis    500
correlated   high_variance  euclidean      500
                            mahalanobis    500
             low_variance   euclidean      500
                            mahalanobis    500
             mixed          euclidean      500
                            mahalanobis    500
isotropic    high_variance  euclidean      500
                            mahalanobis    500
             low_variance   euclidean      500
                            mahalanobis    500
             mixed          euclidean      500
                         

In [14]:
# Experiment 15 — Step 15: Multi-Seed Reference and Calibration

import numpy as np
import pandas as pd
import json

N_SEEDS = CONFIG["n_seeds"]
DIM = CONFIG["embedding_dimension"]
N_REFERENCE = CONFIG["reference_samples"]
N_CALIBRATION = CONFIG["calibration_windows"]
WINDOW_SIZE = CONFIG["window_size"]
TARGET_FPR = CONFIG["target_false_alarm_rate"]
BASE_SEED = CONFIG["base_seed"]

multiseed_models = {}
multiseed_thresholds = {}
multiseed_reference_diagnostics = []
multiseed_calibration_rows = []
multiseed_threshold_rows = []

for seed_index in range(N_SEEDS):
    seed_id = BASE_SEED + 1000 + seed_index

    for condition_index, condition in enumerate(CONFIG["reference_conditions"]):
        covariance = covariances[condition]

        # Separate deterministic random streams for reference and calibration.
        reference_rng = np.random.default_rng(
            seed_id + condition_index * 100
        )
        calibration_rng = np.random.default_rng(
            seed_id + 500 + condition_index * 100
        )

        reference = reference_rng.multivariate_normal(
            mean=np.zeros(DIM),
            cov=covariance,
            size=N_REFERENCE,
            check_valid="raise"
        )

        models = fit_detectors(reference)

        calibration_windows = calibration_rng.multivariate_normal(
            mean=np.zeros(DIM),
            cov=covariance,
            size=N_CALIBRATION * WINDOW_SIZE,
            check_valid="raise"
        ).reshape(N_CALIBRATION, WINDOW_SIZE, DIM)

        scores = score_windows(calibration_windows, models)

        key = (seed_index, condition)
        multiseed_models[key] = models
        multiseed_thresholds[key] = {}

        for detector in CONFIG["detectors"]:
            detector_scores = scores[detector]

            try:
                threshold = np.quantile(
                    detector_scores, 1 - TARGET_FPR, method="higher"
                )
            except TypeError:
                threshold = np.quantile(
                    detector_scores,
                    1 - TARGET_FPR,
                    interpolation="higher"
                )

            threshold = float(threshold)
            multiseed_thresholds[key][detector] = threshold

            multiseed_threshold_rows.append({
                "seed_index": seed_index,
                "seed_id": seed_id,
                "condition": condition,
                "detector": detector,
                "target_fpr": TARGET_FPR,
                "threshold": threshold,
                "calibration_alarm_rate": float(
                    np.mean(detector_scores > threshold)
                ),
            })

            for window_index, score in enumerate(detector_scores):
                multiseed_calibration_rows.append({
                    "seed_index": seed_index,
                    "seed_id": seed_id,
                    "condition": condition,
                    "detector": detector,
                    "window_index": window_index,
                    "score": float(score),
                    "threshold": threshold,
                })

        multiseed_reference_diagnostics.append({
            "seed_index": seed_index,
            "seed_id": seed_id,
            "condition": condition,
            "reference_samples": len(reference),
            "dimensions": reference.shape[1],
            "all_finite": bool(np.isfinite(reference).all()),
            "mean_abs_average": float(
                np.abs(reference.mean(axis=0)).mean()
            ),
            "minimum_empirical_eigenvalue": float(
                np.linalg.eigvalsh(np.cov(reference, rowvar=False)).min()
            ),
        })

multiseed_reference_df = pd.DataFrame(multiseed_reference_diagnostics)
multiseed_calibration_df = pd.DataFrame(multiseed_calibration_rows)
multiseed_threshold_df = pd.DataFrame(multiseed_threshold_rows)

multiseed_reference_df.to_csv(
    OUTPUT_DIR / "multiseed_reference_diagnostics.csv", index=False
)
multiseed_calibration_df.to_csv(
    OUTPUT_DIR / "multiseed_calibration_scores.csv", index=False
)
multiseed_threshold_df.to_csv(
    OUTPUT_DIR / "multiseed_frozen_thresholds.csv", index=False
)

print("Reference datasets:", len(multiseed_reference_df))
print("Calibration score rows:", len(multiseed_calibration_df))
print("Frozen threshold rows:", len(multiseed_threshold_df))

print("\nThreshold summary:")
print(
    multiseed_threshold_df.groupby(
        ["condition", "detector"]
    )["threshold"].agg(["mean", "std", "min", "max"]).round(6).to_string()
)

assert len(multiseed_reference_df) == N_SEEDS * 3
assert len(multiseed_calibration_df) == N_SEEDS * 3 * 2 * N_CALIBRATION
assert len(multiseed_threshold_df) == N_SEEDS * 3 * 2
assert multiseed_reference_df["all_finite"].all()
assert np.isfinite(multiseed_calibration_df["score"]).all()
assert np.isfinite(multiseed_threshold_df["threshold"]).all()

print("\nMulti-seed reference/calibration validation: PASSED")

Reference datasets: 60
Calibration score rows: 24000
Frozen threshold rows: 120

Threshold summary:
                             mean       std       min       max
condition   detector                                           
anisotropic euclidean    2.513004  0.045689  2.442170  2.594950
            mahalanobis  2.058223  0.025095  2.012306  2.117857
correlated  euclidean    2.191161  0.044874  2.134683  2.290475
            mahalanobis  2.076873  0.027561  2.020126  2.113246
isotropic   euclidean    2.068919  0.023727  2.023372  2.111994
            mahalanobis  2.070804  0.023029  2.032763  2.121002

Multi-seed reference/calibration validation: PASSED


In [15]:
# Experiment 15 — Step 16: Multi-Seed Independent Control Evaluation

import numpy as np
import pandas as pd

N_SEEDS = CONFIG["n_seeds"]
N_CONTROL = CONFIG["control_windows_per_seed"]
WINDOW_SIZE = CONFIG["window_size"]
DIM = CONFIG["embedding_dimension"]
BASE_SEED = CONFIG["base_seed"]

multiseed_control_rows = []

for seed_index in range(N_SEEDS):
    seed_id = BASE_SEED + 1000 + seed_index

    for condition_index, condition in enumerate(CONFIG["reference_conditions"]):
        covariance = covariances[condition]
        key = (seed_index, condition)

        control_rng = np.random.default_rng(
            seed_id + 800 + condition_index * 100
        )

        control_windows = control_rng.multivariate_normal(
            mean=np.zeros(DIM),
            cov=covariance,
            size=N_CONTROL * WINDOW_SIZE,
            check_valid="raise"
        ).reshape(N_CONTROL, WINDOW_SIZE, DIM)

        scores = score_windows(
            control_windows,
            multiseed_models[key]
        )

        for detector in CONFIG["detectors"]:
            threshold = multiseed_thresholds[key][detector]
            detector_scores = scores[detector]
            alarms = detector_scores > threshold

            for window_index, (score, alarm) in enumerate(
                zip(detector_scores, alarms)
            ):
                multiseed_control_rows.append({
                    "seed_index": seed_index,
                    "seed_id": seed_id,
                    "condition": condition,
                    "detector": detector,
                    "window_index": window_index,
                    "score": float(score),
                    "threshold": threshold,
                    "alarm": bool(alarm),
                })

multiseed_control_df = pd.DataFrame(multiseed_control_rows)

control_path = OUTPUT_DIR / "multiseed_independent_control_results.csv"
multiseed_control_df.to_csv(control_path, index=False)

# First aggregate within each seed, then summarize across seeds.
seed_control_summary = (
    multiseed_control_df
    .groupby(["seed_index", "condition", "detector"])
    .agg(
        control_windows=("alarm", "size"),
        false_alarms=("alarm", "sum"),
        observed_fpr=("alarm", "mean"),
    )
    .reset_index()
)

seed_summary_path = OUTPUT_DIR / "multiseed_control_seed_summary.csv"
seed_control_summary.to_csv(seed_summary_path, index=False)

control_summary = (
    seed_control_summary
    .groupby(["condition", "detector"])
    .agg(
        seeds=("seed_index", "nunique"),
        mean_fpr=("observed_fpr", "mean"),
        sd_fpr=("observed_fpr", "std"),
        min_fpr=("observed_fpr", "min"),
        max_fpr=("observed_fpr", "max"),
        total_false_alarms=("false_alarms", "sum"),
        total_windows=("control_windows", "sum"),
    )
    .reset_index()
)

control_summary["pooled_fpr"] = (
    control_summary["total_false_alarms"]
    / control_summary["total_windows"]
)

summary_path = OUTPUT_DIR / "multiseed_control_summary.csv"
control_summary.to_csv(summary_path, index=False)

print("Multi-seed independent control summary:")
print(control_summary.round(6).to_string(index=False))

print("\nControl result rows:", len(multiseed_control_df))
print("Seed-level summary rows:", len(seed_control_summary))

assert len(multiseed_control_df) == N_SEEDS * 3 * 2 * N_CONTROL
assert len(seed_control_summary) == N_SEEDS * 3 * 2
assert multiseed_control_df["score"].map(np.isfinite).all()
assert multiseed_control_df["threshold"].map(np.isfinite).all()

print("\nMulti-seed control validation: PASSED")
print("Saved:", control_path)
print("Saved:", seed_summary_path)
print("Saved:", summary_path)

Multi-seed independent control summary:
  condition    detector  seeds  mean_fpr   sd_fpr  min_fpr  max_fpr  total_false_alarms  total_windows  pooled_fpr
anisotropic   euclidean     20    0.0486 0.018389    0.024    0.082                 486          10000      0.0486
anisotropic mahalanobis     20    0.0495 0.014140    0.028    0.082                 495          10000      0.0495
 correlated   euclidean     20    0.0523 0.016686    0.014    0.076                 523          10000      0.0523
 correlated mahalanobis     20    0.0469 0.016727    0.022    0.080                 469          10000      0.0469
  isotropic   euclidean     20    0.0454 0.016785    0.020    0.086                 454          10000      0.0454
  isotropic mahalanobis     20    0.0454 0.016785    0.020    0.086                 454          10000      0.0454

Control result rows: 60000
Seed-level summary rows: 120

Multi-seed control validation: PASSED
Saved: /content/results/experiment_15/multiseed_independent

In [16]:
# Experiment 15 — Step 17: Multi-Seed Drift Evaluation

import numpy as np
import pandas as pd

N_SEEDS = CONFIG["n_seeds"]
N_EVAL = CONFIG["evaluation_windows_per_condition_per_seed"]
WINDOW_SIZE = CONFIG["window_size"]
DIM = CONFIG["embedding_dimension"]
SHIFT_STRENGTH = CONFIG["mean_shift_strength"]
BASE_SEED = CONFIG["base_seed"]

multiseed_evaluation_rows = []

for seed_index in range(N_SEEDS):
    seed_id = BASE_SEED + 1000 + seed_index

    for condition_index, condition in enumerate(CONFIG["reference_conditions"]):
        covariance = covariances[condition]
        key = (seed_index, condition)
        models = multiseed_models[key]

        for direction_index, direction_name in enumerate(CONFIG["drift_directions"]):
            direction = drift_directions[condition][direction_name]
            mean_shift = SHIFT_STRENGTH * direction

            # Independent evaluation stream for each seed, condition, direction.
            eval_seed = (
                seed_id + 1000
                + condition_index * 100
                + direction_index * 10
            )
            eval_rng = np.random.default_rng(eval_seed)

            windows = eval_rng.multivariate_normal(
                mean=mean_shift,
                cov=covariance,
                size=N_EVAL * WINDOW_SIZE,
                check_valid="raise"
            ).reshape(N_EVAL, WINDOW_SIZE, DIM)

            scores = score_windows(windows, models)

            for detector in CONFIG["detectors"]:
                threshold = multiseed_thresholds[key][detector]
                detector_scores = scores[detector]
                alarms = detector_scores > threshold

                for window_index, (score, alarm) in enumerate(
                    zip(detector_scores, alarms)
                ):
                    multiseed_evaluation_rows.append({
                        "seed_index": seed_index,
                        "seed_id": seed_id,
                        "condition": condition,
                        "direction": direction_name,
                        "detector": detector,
                        "evaluation_seed": eval_seed,
                        "window_index": window_index,
                        "shift_strength": SHIFT_STRENGTH,
                        "score": float(score),
                        "threshold": threshold,
                        "alarm": bool(alarm),
                    })

multiseed_evaluation_df = pd.DataFrame(multiseed_evaluation_rows)

evaluation_path = OUTPUT_DIR / "multiseed_drift_evaluation_results.csv"
multiseed_evaluation_df.to_csv(evaluation_path, index=False)

# Calculate detection rate within each seed first.
seed_evaluation_summary = (
    multiseed_evaluation_df
    .groupby(["seed_index", "condition", "direction", "detector"])
    .agg(
        evaluation_windows=("alarm", "size"),
        detected_windows=("alarm", "sum"),
        detection_rate=("alarm", "mean"),
    )
    .reset_index()
)

seed_summary_path = OUTPUT_DIR / "multiseed_drift_seed_summary.csv"
seed_evaluation_summary.to_csv(seed_summary_path, index=False)

# Summarize the distribution across 20 seed-level rates.
evaluation_summary = (
    seed_evaluation_summary
    .groupby(["condition", "direction", "detector"])
    .agg(
        seeds=("seed_index", "nunique"),
        mean_detection_rate=("detection_rate", "mean"),
        sd_detection_rate=("detection_rate", "std"),
        min_detection_rate=("detection_rate", "min"),
        max_detection_rate=("detection_rate", "max"),
        total_detected=("detected_windows", "sum"),
        total_windows=("evaluation_windows", "sum"),
    )
    .reset_index()
)

evaluation_summary["pooled_detection_rate"] = (
    evaluation_summary["total_detected"]
    / evaluation_summary["total_windows"]
)

summary_path = OUTPUT_DIR / "multiseed_drift_evaluation_summary.csv"
evaluation_summary.to_csv(summary_path, index=False)

print("Multi-seed drift evaluation summary:")
print(evaluation_summary.round(6).to_string(index=False))

print("\nEvaluation result rows:", len(multiseed_evaluation_df))
print("Seed-level summary rows:", len(seed_evaluation_summary))

assert len(multiseed_evaluation_df) == (
    N_SEEDS * 3 * 3 * 2 * N_EVAL
)
assert len(seed_evaluation_summary) == N_SEEDS * 3 * 3 * 2
assert multiseed_evaluation_df["score"].map(np.isfinite).all()
assert multiseed_evaluation_df["threshold"].map(np.isfinite).all()

print("\nMulti-seed drift evaluation validation: PASSED")
print("Saved:", evaluation_path)
print("Saved:", seed_summary_path)
print("Saved:", summary_path)

Multi-seed drift evaluation summary:
  condition     direction    detector  seeds  mean_detection_rate  sd_detection_rate  min_detection_rate  max_detection_rate  total_detected  total_windows  pooled_detection_rate
anisotropic high_variance   euclidean     20               0.0797           0.027371               0.024               0.134             797          10000                 0.0797
anisotropic high_variance mahalanobis     20               0.0615           0.021559               0.026               0.124             615          10000                 0.0615
anisotropic  low_variance   euclidean     20               0.0691           0.028584               0.034               0.140             691          10000                 0.0691
anisotropic  low_variance mahalanobis     20               0.2988           0.060484               0.176               0.436            2988          10000                 0.2988
anisotropic         mixed   euclidean     20               0.0730   

In [17]:
# Experiment 15 — Step 18: Across-Seed Detector Comparison

import numpy as np
import pandas as pd

seed_pivot = seed_evaluation_summary.pivot(
    index=["seed_index", "condition", "direction"],
    columns="detector",
    values="detection_rate"
).reset_index()

seed_pivot["mahalanobis_minus_euclidean"] = (
    seed_pivot["mahalanobis"] - seed_pivot["euclidean"]
)

seed_pivot["absolute_difference"] = (
    seed_pivot["mahalanobis_minus_euclidean"].abs()
)

seed_comparison_path = OUTPUT_DIR / "multiseed_detection_rate_differences.csv"
seed_pivot.to_csv(seed_comparison_path, index=False)

difference_summary = (
    seed_pivot
    .groupby(["condition", "direction"])
    .agg(
        seeds=("seed_index", "nunique"),
        mean_euclidean_rate=("euclidean", "mean"),
        mean_mahalanobis_rate=("mahalanobis", "mean"),
        mean_difference=("mahalanobis_minus_euclidean", "mean"),
        sd_difference=("mahalanobis_minus_euclidean", "std"),
        min_difference=("mahalanobis_minus_euclidean", "min"),
        max_difference=("mahalanobis_minus_euclidean", "max"),
        seeds_mahalanobis_higher=(
            "mahalanobis_minus_euclidean",
            lambda x: int((x > 0).sum())
        ),
        seeds_euclidean_higher=(
            "mahalanobis_minus_euclidean",
            lambda x: int((x < 0).sum())
        ),
        seeds_tied=(
            "mahalanobis_minus_euclidean",
            lambda x: int((x == 0).sum())
        ),
    )
    .reset_index()
)

difference_summary_path = OUTPUT_DIR / "multiseed_detection_difference_summary.csv"
difference_summary.to_csv(difference_summary_path, index=False)

print("Across-seed detector comparison:")
print(difference_summary.round(6).to_string(index=False))

assert len(seed_pivot) == 20 * 3 * 3
assert (difference_summary["seeds"] == 20).all()
assert np.isfinite(seed_pivot["mahalanobis_minus_euclidean"]).all()

print("\nAcross-seed comparison validation: PASSED")
print("Saved:", seed_comparison_path)
print("Saved:", difference_summary_path)

Across-seed detector comparison:
  condition     direction  seeds  mean_euclidean_rate  mean_mahalanobis_rate  mean_difference  sd_difference  min_difference  max_difference  seeds_mahalanobis_higher  seeds_euclidean_higher  seeds_tied
anisotropic high_variance     20               0.0797                 0.0615          -0.0182       0.020864          -0.060           0.020                         4                      16           0
anisotropic  low_variance     20               0.0691                 0.2988           0.2297       0.049368           0.140           0.308                        20                       0           0
anisotropic         mixed     20               0.0730                 0.1587           0.0857       0.033899           0.032           0.146                        20                       0           0
 correlated high_variance     20               0.0969                 0.0624          -0.0345       0.025653          -0.088           0.000               

In [18]:
# Experiment 15 — Step 19: Confidence Intervals Across Seeds

import numpy as np
import pandas as pd
from scipy.stats import t

rows = []

for (condition, direction), group in seed_pivot.groupby(
    ["condition", "direction"]
):
    differences = group["mahalanobis_minus_euclidean"].to_numpy()
    n = len(differences)

    mean_diff = differences.mean()
    sd_diff = differences.std(ddof=1)
    se_diff = sd_diff / np.sqrt(n)
    critical_value = t.ppf(0.975, df=n - 1)

    ci_lower = mean_diff - critical_value * se_diff
    ci_upper = mean_diff + critical_value * se_diff

    rows.append({
        "condition": condition,
        "direction": direction,
        "seeds": n,
        "mean_difference": mean_diff,
        "sd_difference": sd_diff,
        "ci95_lower": ci_lower,
        "ci95_upper": ci_upper,
        "ci_excludes_zero": bool(
            ci_lower > 0 or ci_upper < 0
        ),
    })

confidence_df = pd.DataFrame(rows).sort_values(
    ["condition", "direction"]
).reset_index(drop=True)

confidence_path = OUTPUT_DIR / "multiseed_detection_difference_ci95.csv"
confidence_df.to_csv(confidence_path, index=False)

print("95% confidence intervals for Mahalanobis − Euclidean:")
print(confidence_df.round(6).to_string(index=False))

print("\nSaved:", confidence_path)
print("Confidence interval validation: PASSED")

95% confidence intervals for Mahalanobis − Euclidean:
  condition     direction  seeds  mean_difference  sd_difference  ci95_lower  ci95_upper  ci_excludes_zero
anisotropic high_variance     20          -0.0182       0.020864   -0.027965   -0.008435              True
anisotropic  low_variance     20           0.2297       0.049368    0.206595    0.252805              True
anisotropic         mixed     20           0.0857       0.033899    0.069835    0.101565              True
 correlated high_variance     20          -0.0345       0.025653   -0.046506   -0.022494              True
 correlated  low_variance     20           0.3163       0.072415    0.282409    0.350191              True
 correlated         mixed     20           0.1141       0.046821    0.092187    0.136013              True
  isotropic high_variance     20           0.0000       0.001298   -0.000607    0.000607             False
  isotropic  low_variance     20           0.0003       0.000979   -0.000158    0.000758  

In [19]:
# Experiment 15 — Step 20: Final Integrity Audit

import numpy as np
import pandas as pd
import json

N_SEEDS = CONFIG["n_seeds"]
N_CONTROL = CONFIG["control_windows_per_seed"]
N_EVAL = CONFIG["evaluation_windows_per_condition_per_seed"]

audit_checks = []

def record_check(name, passed, detail):
    audit_checks.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail),
    })

# 1. Reference and calibration coverage
record_check(
    "Reference datasets",
    len(multiseed_reference_df) == N_SEEDS * 3,
    f"{len(multiseed_reference_df)} rows; expected {N_SEEDS * 3}"
)

record_check(
    "Calibration scores",
    len(multiseed_calibration_df) == N_SEEDS * 3 * 2 * CONFIG["calibration_windows"],
    f"{len(multiseed_calibration_df)} rows"
)

record_check(
    "Frozen thresholds",
    len(multiseed_threshold_df) == N_SEEDS * 3 * 2,
    f"{len(multiseed_threshold_df)} rows"
)

# 2. Independent control coverage
record_check(
    "Independent control rows",
    len(multiseed_control_df) == N_SEEDS * 3 * 2 * N_CONTROL,
    f"{len(multiseed_control_df)} rows"
)

record_check(
    "Control seed-level summaries",
    len(seed_control_summary) == N_SEEDS * 3 * 2,
    f"{len(seed_control_summary)} rows"
)

# 3. Drift evaluation coverage
record_check(
    "Drift evaluation rows",
    len(multiseed_evaluation_df) == N_SEEDS * 3 * 3 * 2 * N_EVAL,
    f"{len(multiseed_evaluation_df)} rows"
)

record_check(
    "Drift seed-level summaries",
    len(seed_evaluation_summary) == N_SEEDS * 3 * 3 * 2,
    f"{len(seed_evaluation_summary)} rows"
)

record_check(
    "Across-seed comparison rows",
    len(seed_pivot) == N_SEEDS * 3 * 3,
    f"{len(seed_pivot)} rows"
)

record_check(
    "Confidence interval rows",
    len(confidence_df) == 9,
    f"{len(confidence_df)} condition-direction rows"
)

# 4. Numeric validity
numeric_checks = {
    "Calibration scores finite": np.isfinite(
        multiseed_calibration_df["score"]
    ).all(),
    "Control scores finite": np.isfinite(
        multiseed_control_df["score"]
    ).all(),
    "Evaluation scores finite": np.isfinite(
        multiseed_evaluation_df["score"]
    ).all(),
    "Frozen thresholds finite": np.isfinite(
        multiseed_threshold_df["threshold"]
    ).all(),
    "Detection rates within [0, 1]": (
        multiseed_evaluation_df["alarm"].isin([True, False]).all()
    ),
}

for name, passed in numeric_checks.items():
    record_check(name, passed, "Validated")

# 5. Threshold consistency
threshold_check = True

for row in multiseed_threshold_df.itertuples(index=False):
    key = (row.seed_index, row.condition)
    expected = multiseed_thresholds[key][row.detector]

    if not np.isclose(row.threshold, expected):
        threshold_check = False
        break

record_check(
    "Threshold consistency",
    threshold_check,
    "Saved thresholds match the thresholds used by the evaluation"
)

# 6. Save audit report
audit_df = pd.DataFrame(audit_checks)
audit_path = OUTPUT_DIR / "experiment_15_integrity_audit.csv"
audit_df.to_csv(audit_path, index=False)

print("Final integrity audit:")
print(audit_df.to_string(index=False))

all_passed = audit_df["passed"].all()
print("\nOverall audit:", "PASSED" if all_passed else "FAILED")
print("Saved:", audit_path)

assert all_passed, "One or more integrity checks failed."

Final integrity audit:
                        check  passed                                                       detail
           Reference datasets    True                                         60 rows; expected 60
           Calibration scores    True                                                   24000 rows
            Frozen thresholds    True                                                     120 rows
     Independent control rows    True                                                   60000 rows
 Control seed-level summaries    True                                                     120 rows
        Drift evaluation rows    True                                                  180000 rows
   Drift seed-level summaries    True                                                     360 rows
  Across-seed comparison rows    True                                                     180 rows
     Confidence interval rows    True                                   9 condition-di

In [20]:
# Experiment 15 — Package all results for download

from pathlib import Path
import zipfile
from google.colab import files

RESULTS_DIR = Path("/content/results/experiment_15")
ZIP_PATH = Path("/content/experiment_15_results.zip")

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"Results folder not found: {RESULTS_DIR}"
    )

result_files = sorted(
    path for path in RESULTS_DIR.rglob("*")
    if path.is_file()
)

if not result_files:
    raise RuntimeError("No result files found to package.")

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zip_file:
    for path in result_files:
        zip_file.write(
            path,
            arcname=str(Path("experiment_15") / path.relative_to(RESULTS_DIR))
        )

print("Files packaged:", len(result_files))
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024 ** 2), 2))
print("ZIP created:", ZIP_PATH)

files.download(str(ZIP_PATH))

Files packaged: 27
ZIP size (MB): 3.5
ZIP created: /content/experiment_15_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>